In [1]:
import os

os.environ["OPENAI_API_KEY"] = "sk-"

In [2]:
import json
from uuid import uuid4
from textwrap import dedent

import numpy as np
import pandas as pd

from pydantic import BaseModel
from openai import OpenAI
from openai.lib._pydantic import to_strict_json_schema
from rich import print as print_pretty

## Data Loading

In [3]:
df = pd.read_excel("../dataset/data-corpus-base.xlsx", sheet_name="BEIR Corpus")
df = df.dropna()

df.head()

,Kode Doc,Kode Dokumen,Judul,Konten
0,doc1,b2044980-d3a0-4cf5-91b7-69cda3b7aafe,Wakil Dubes Walanda Gumbira Ningali Holland In...,bogor hotél institut (bhi) gawé bareng jeung f...
1,doc2,3fe1a195-5081-4855-95fa-cba5eff1cdcf,Warga Bogor Mapag Taun anyar Islam,bogor - datang ton anyar islam 1431-hijréh pap...
2,doc3,c121c9b4-a97a-4ba4-8f5e-5e58578b17ce,Warugan Lemah: Pola Lembur Urang Sunda Buhun,naskah warugan lemah kandelna ngan tilu lempir...
3,doc4,b87199ed-ca2f-4e2b-a323-23f3b387a9aa,Manfaat Olahraga Pikeun Kasehatan,anu ku urang tos terang olahraga teh penting p...
4,doc5,cc6a7052-6620-46cc-8704-9f54e3e99016,DINA JANDÉLA INDUNG,"méméh layung kubur panineungan dina jandéla, g..."


In [4]:
df.iloc[0, 3]

'bogor hotél institut (bhi) gawé bareng jeung forum indonesé-nederland (fined), sarta hotél salak the heritage gel "holland indonésé festival" di hotél salak, jalan ir jonda, kota bogor, (15/11/2009). festival anu buka wakil duta besar walanda keur indonésé, annemieke ruigrok boga tuju pikeun ngawanohkeun budaya anu dipiboga ku do nagara ieu ka masarakat. wakil duta besar walanda, annemieke ruigrok kaku pohara gumbira ku lumangsungna acara ieu. “indonésé jeung walanda boga hubungan lit,“ ceuk manéhna. "sanajan walanda boga sajarah mangsa ka tukang anu kurang alus di mata masarakat indonésé, tapi henteu ngaleungitkeun hubungan eta," sambungna. lamun tempo sajarah mangsa ka tukang indonésé jeung walanda boga sawatara kamiripan dina hal kabudayan. teu eutik warga walanda anu mikaresep seni budaya anu asalna ti indonésé. "urang walanda pohara resep ku masak urang indonésé," terus dina basa walanda. "kuring taji kasoméahanana rahayat indonésé. kuring gumbira aya di indonésé nu jalma saroméa

## Synthetic Data Generation using LLMs

In [5]:
client = OpenAI()

In [6]:
SYSTEM_PROMPTS = {
    "BEIR": dedent("""
                    Generate 5 short search query-answer pairs based on the provided document. Each query should resemble a natural search input (as if searching on Google or other search engines).
                    - The query and the answer must be written in Sundanese.
                    - Ensure the response is concise, accurate, and directly relevant to the document.
                    """).strip(),
    "MSMARCO": dedent("""
                    Create an MS-MARCO triplet dataset from the given document. Each triplet consists of:

                    1. Query: A short, natural search query based on the document.
                    2. Positive Passage: A passage from the document that directly answers the query.
                    3. Negative Passage: A passage from the document that does not answer the query but is still related to the topic.

                    Requirements:

                    - Generate exactly 5 triplets.
                    - Write all elements (query, positive passage, negative passage) in Sundanese.
                    - Ensure the query and passages are concise and coherent.
                    """).strip(),
}

### Synthetic BEIR

In [7]:
class BEIRQueryItem(BaseModel):
    search_term: str
    answer: str


class BEIRQuery(BaseModel):
    queries: list[BEIRQueryItem]

In [8]:
completion_beir = client.beta.chat.completions.parse(
    model="gpt-4o",
    response_format=BEIRQuery,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["BEIR"],
        },
        {
            "role": "user",
            "content": df.iloc[0, 3],
        },
    ],
)

In [9]:
print_pretty(completion_beir)

ParsedChatCompletion[BEIRQuery](
    id='chatcmpl-AjIPQEbzvHXeCAjT2FaNpSEUpfjEP',
    choices=[
        ParsedChoice[BEIRQuery](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[BEIRQuery](
                content='{"queries":[{"search_term":"Naon tujuan Holland Indonesé Festival di 
Bogor?","answer":"Tujuan Holland Indonesé Festival di Bogor nyaéta pikeun ngawanohkeun budaya anu dipiboga ku do 
nagara ieu ka masarakat."},{"search_term":"Kapan Holland Indonesé Festival dihotél Salak jalan Ir H Juanda Kota 
Bogor?","answer":"Holland Indonesé Festival dihotél Salak jalan Ir H Juanda Kota Bogor dilaksanakeun dina 15 
Nopémber 2009."},{"search_term":"Saha anu ngabuka Holland Indonesé Festival di Bogor?","answer":"Wakil Duta Besar 
Walanda keur Indonésé, Annemieke Ruigrok, anu ngabuka festival éta."},{"search_term":"Naon parabot seni anu 
dihadirkeun dina festival di Bogor?","answer":"Parabot seni anu dihadirkeun diantarana batik jeung seni 
angklung."},{"search_term":"Saha R. Ay. Suni Wijogawati dina acara festival di Bogor?","answer":"R. Ay. Suni 
Wijogawati mangrupa wakil pupuhu Fined, oge management assitant Erasmus Huis."}]}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=[],
                parsed=BEIRQuery(
                    queries=[
                        BEIRQueryItem(
                            search_term='Naon tujuan Holland Indonesé Festival di Bogor?',
                            answer='Tujuan Holland Indonesé Festival di Bogor nyaéta pikeun ngawanohkeun budaya anu
dipiboga ku do nagara ieu ka masarakat.'
                        ),
                        BEIRQueryItem(
                            search_term='Kapan Holland Indonesé Festival dihotél Salak jalan Ir H Juanda Kota 
Bogor?',
                            answer='Holland Indonesé Festival dihotél Salak jalan Ir H Juanda Kota Bogor 
dilaksanakeun dina 15 Nopémber 2009.'
                        ),
                        BEIRQueryItem(
                            search_term='Saha anu ngabuka Holland Indonesé Festival di Bogor?',
                            answer='Wakil Duta Besar Walanda keur Indonésé, Annemieke Ruigrok, anu ngabuka festival
éta.'
                        ),
                        BEIRQueryItem(
                            search_term='Naon parabot seni anu dihadirkeun dina festival di Bogor?',
                            answer='Parabot seni anu dihadirkeun diantarana batik jeung seni angklung.'
                        ),
                        BEIRQueryItem(
                            search_term='Saha R. Ay. Suni Wijogawati dina acara festival di Bogor?',
                            answer='R. Ay. Suni Wijogawati mangrupa wakil pupuhu Fined, oge management assitant 
Erasmus Huis.'
                        )
                    ]
                )
            )
        )
    ],
    created=1735358888,
    model='gpt-4o-2024-08-06',
    object='chat.completion',
    service_tier=None,
    system_fingerprint='fp_d28bcae782',
    usage=CompletionUsage(
        completion_tokens=253,
        prompt_tokens=643,
        total_tokens=896,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=0
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)

### Synthetic MS-MARCO Triplet

In [10]:
class MSMARCOTripletItem(BaseModel):
    query: str
    positive_passage: str
    negative_passage: str


class MSMARCO(BaseModel):
    triplets: list[MSMARCOTripletItem]

In [11]:
completion_marco = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=MSMARCO,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["MSMARCO"],
        },
        {
            "role": "user",
            "content": df.iloc[0, 3],
        },
    ],
)

In [12]:
print_pretty(completion_marco)

ParsedChatCompletion[MSMARCO](
    id='chatcmpl-AjIPTEEzKgiuGD34ysckAKyrDAkik',
    choices=[
        ParsedChoice[MSMARCO](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[MSMARCO](
                content='{"triplets":[{"query":"Naon tujuan festival budaya di Bogor?","positive_passage":"Festival
anu buka wakil duta besar walanda keur indonésé, annemieke ruigrok boga tuju pikeun ngawanohkeun budaya anu 
dipiboga ku do nagara ieu ka masarakat.","negative_passage":"Sagala rupa budaya milik bangsa indonésé dihadirkan 
dina festival ieu, antara lain batik jeung seni angklung."},{"query":"Saha anu ngadatang acara 
kasebut?","positive_passage":"Festival anu buka wakil duta besar walanda keur indonésé, annemieke ruigrok boga tuju
untuk ngawanohkeun budaya anu dipiboga ku do nagara ieu ka masarakat.","negative_passage":"Ruigrok sempet nengetan 
pengrajin batik mraktékkeun cara sieun batik tulis."},{"query":"Kumaha reaksi wakil duta besar ngeunaan acara 
festival?","positive_passage":"Wakil duta besar walanda, annemieke ruigrok kaku pohara gumbira ku lumangsungna 
acara ieu.","negative_passage":"Nurutkeun r. ay. suni wijogawati laku wakil pupuhu fined, festival budaya ieu ngan 
gel sapoé."},{"query":"Sabaraha kali festival ieu diayakeun di kota séjén?","positive_passage":"Saméméh laksana di 
kota bogor, kagétan rupa ogé geus laksana di kota-kota séjén.","negative_passage":"Kuring gumbira aya di indonésé 
nu jalma saroméah, ceuk deui."},{"query":"Seni budaya naon anu ditampilkeun dina 
festival?","positive_passage":"Sagala rupa budaya milik bangsa indonésé dihadirkan dina festival ieu, antara lain 
batik jeung seni angklung.","negative_passage":"Urang walanda pohara resep ku masak urang indonésé, terus dina basa
walanda."}]}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=[],
                parsed=MSMARCO(
                    triplets=[
                        MSMARCOTripletItem(
                            query='Naon tujuan festival budaya di Bogor?',
                            positive_passage='Festival anu buka wakil duta besar walanda keur indonésé, annemieke 
ruigrok boga tuju pikeun ngawanohkeun budaya anu dipiboga ku do nagara ieu ka masarakat.',
                            negative_passage='Sagala rupa budaya milik bangsa indonésé dihadirkan dina festival 
ieu, antara lain batik jeung seni angklung.'
                        ),
                        MSMARCOTripletItem(
                            query='Saha anu ngadatang acara kasebut?',
                            positive_passage='Festival anu buka wakil duta besar walanda keur indonésé, annemieke 
ruigrok boga tuju untuk ngawanohkeun budaya anu dipiboga ku do nagara ieu ka masarakat.',
                            negative_passage='Ruigrok sempet nengetan pengrajin batik mraktékkeun cara sieun batik 
tulis.'
                        ),
                        MSMARCOTripletItem(
                            query='Kumaha reaksi wakil duta besar ngeunaan acara festival?',
                            positive_passage='Wakil duta besar walanda, annemieke ruigrok kaku pohara gumbira ku 
lumangsungna acara ieu.',
                            negative_passage='Nurutkeun r. ay. suni wijogawati laku wakil pupuhu fined, festival 
budaya ieu ngan gel sapoé.'
                        ),
                        MSMARCOTripletItem(
                            query='Sabaraha kali festival ieu diayakeun di kota séjén?',
                            positive_passage='Saméméh laksana di kota bogor, kagétan rupa ogé geus laksana di 
kota-kota séjén.',
                            negative_passage='Kuring gumbira aya di indonésé nu jalma saroméah, ceuk deui.'
                        ),
                        MSMARCOTripletItem(
                            query='Seni budaya naon anu ditampilkeun dina festi

## Generate OpenAI Batch Request

In [13]:
def generate_batch(df: pd.DataFrame, system_prompt: str, base_model: BaseModel):
    for row in df.itertuples():
        id = str(uuid4())
        yield (
            id,
            row[2],
            {
                "custom_id": id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": "gpt-4o-mini",
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": row.Konten},
                    ],
                    "response_format": {
                        "type": "json_schema",
                        "json_schema": {
                            "name": base_model.__name__,
                            "strict": True,
                            "schema": to_strict_json_schema(base_model),
                        },
                    },
                },
            },
        )

In [14]:
def submit_batch_beir():
    with open(f"beir_batch.jsonl", "w") as fm, open("beir_map.jsonl", "w") as mm:
        for custom_id, doc_id, item in generate_batch(
            df, SYSTEM_PROMPTS["BEIR"], BEIRQuery
        ):
            json.dump(item, fm)
            fm.write("\n")

            json.dump({"custom_id": custom_id, "doc_id": doc_id}, mm)
            mm.write("\n")

    batch_file = client.files.create(
        file=open("beir_batch.jsonl", "rb"), purpose="batch"
    )

    return client.batches.create(
        input_file_id=batch_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )

In [15]:
batch_job_beir = submit_batch_beir()
print_pretty(batch_job_beir)

Batch(
    id='batch_676f79b4369081909de9dd17e0ecc666',
    completion_window='24h',
    created_at=1735358900,
    endpoint='/v1/chat/completions',
    input_file_id='file-7BRnMdoYqQfYyWk1dhFK2p',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1735445300,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=0)
)

In [16]:
def submit_batch_marco():
    with open(f"marco_batch.jsonl", "w") as fm, open("marco_map.jsonl", "w") as mm:
        for custom_id, doc_id, item in generate_batch(
            df, SYSTEM_PROMPTS["MSMARCO"], MSMARCO
        ):
            json.dump(item, fm)
            fm.write("\n")

            json.dump({"custom_id": custom_id, "doc_id": doc_id}, mm)
            mm.write("\n")

    batch_file = client.files.create(
        file=open("marco_batch.jsonl", "rb"), purpose="batch"
    )

    return client.batches.create(
        input_file_id=batch_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )

In [17]:
batch_job_marco = submit_batch_marco()
print_pretty(batch_job_marco)

Batch(
    id='batch_676f79bab1988190b988ae5b09f4ffbf',
    completion_window='24h',
    created_at=1735358906,
    endpoint='/v1/chat/completions',
    input_file_id='file-4EiJjasrQp4ZagnxReq2AU',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1735445306,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=0)
)